In [5]:
import os
from elasticsearch import Elasticsearch
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

es = Elasticsearch(os.getenv("ES_HOST"), basic_auth=(os.getenv("ES_USER"), os.getenv("ES_PASSWORD")))

# Check connection
if es.ping():
    print("Connected to Elasticsearch")
else:
    print("Failed to connect to Elasticsearch")

Connected to Elasticsearch


In [6]:
response = es.search(
    index="gumball_transcripts_with_prompts",
    query={"match_all": {}},
    size=10000
)

documents = response["hits"]["hits"]

print(f"Loaded {len(documents)} documents")

Loaded 256 documents


In [ ]:
from elasticsearch.helpers import bulk

# Calculate word counts for each document
word_counts = []
for doc in documents:
    text = "\n".join([t["text"] for t in doc["_source"]["transcript"]])
    word_count = len(text.split())
    word_counts.append(word_count)

# Sort word counts and calculate percentiles
sorted_word_counts = sorted(word_counts)
p33 = sorted_word_counts[int(0.33 * len(sorted_word_counts))]
p66 = sorted_word_counts[int(0.66 * len(sorted_word_counts))]

# Prepare bulk update actions
actions = []
for doc, wc in zip(documents, word_counts):
    if wc < p33:
        length_category = "short"
    elif wc < p66:
        length_category = "medium"
    else:
        length_category = "long"
    
    # Add the length field to the document
    doc["_source"]["length"] = length_category

    # Prepare the update action
    action = {
        "_op_type": "update",
        "_index": doc["_index"],
        "_id": doc["_id"],
        "doc": {"length": length_category}
    }
    actions.append(action)

# Perform bulk update
bulk(es, actions)

print("Updated documents with length categories.")

Updated documents with length categories.
